<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione5/Lezione5_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 5 — Tools, Function Calling & MCP

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 04/06/2026

---

### 🎯 Obiettivi
- ✅ Definire tool in JSON e integrarli nell'API
- ✅ Implementare il tool loop
- ✅ Costruire 3 tool reali (calcolatrice, meteo, Wikipedia)
- ✅ Capire MCP e connettersi a un server

In [1]:
!pip install anthropic requests -q
from google.colab import userdata
import anthropic, os, json, requests

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.6/838.6 kB 6.1 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Tool Calcolatrice

Il tool più semplice — definizione JSON + funzione Python.

In [2]:
# DEFINIZIONE del tool (cosa il modello può leggere)
tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica. Usa questo tool per calcoli aritmetici precisi.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "L'espressione matematica da calcolare. Es: '234 * 567' o '(100 + 200) / 3'"
            }
        },
        "required": ["espressione"]
    }
}

# IMPLEMENTAZIONE del tool (la funzione Python reale)
def calcola(espressione: str) -> str:
    """Valuta un'espressione matematica in modo sicuro."""
    try:
        # eval() limitato solo a operazioni matematiche
        allowed = set('0123456789+-*/().% ')
        if not all(c in allowed for c in espressione):
            return "Errore: espressione non valida"
        risultato = eval(espressione)
        return f"{espressione} = {risultato}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"

# Test diretto
print(calcola("234 * 567"))
print(calcola("(100 + 200) / 3"))

234 * 567 = 132678
(100 + 200) / 3 = 100.0


In [3]:
# IL TOOL LOOP — il cuore del function calling
def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    return f"Tool '{nome}' non trovato"

def chat_con_tool(messaggio, tools, history=None):
    """Chatbot con tool use. Gestisce il loop automaticamente."""
    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio})

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # Se il modello ha finito, restituisce la risposta
        if response.stop_reason == "end_turn":
            testo = next(b.text for b in response.content if b.type == "text")
            history.append({"role": "assistant", "content": response.content})
            return testo, history

        # Se il modello vuole usare un tool
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})

            # Esegui tutti i tool richiesti
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Chiama tool: {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ Risultato: {risultato}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })

            # Rimanda i risultati al modello
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Domanda: Quanto fa 1234 * 5678?")
risposta, _ = chat_con_tool("Quanto fa 1234 * 5678?", [tool_calcolatrice])
print(f"\n🤖 Risposta: {risposta}")

❓ Domanda: Quanto fa 1234 * 5678?
  🔧 Chiama tool: calcola({'espressione': '1234 * 5678'})
  ✅ Risultato: 1234 * 5678 = 7006652

🤖 Risposta: Il risultato di **1234 × 5678 = 7.006.652**


---
## 2. Tool Meteo (API reale)

Usiamo [open-meteo.com](https://open-meteo.com) — gratuito, senza API key.

In [4]:
# Coordinate delle città sarde principali
CITTA_COORD = {
    "sassari": {"lat": 40.7259, "lon": 8.5563},
    "cagliari": {"lat": 39.2238, "lon": 9.1217},
    "nuoro": {"lat": 40.3207, "lon": 9.3311},
    "oristano": {"lat": 39.9069, "lon": 8.5889},
    "olbia": {"lat": 40.9237, "lon": 9.4992},
    "roma": {"lat": 41.9028, "lon": 12.4964},
    "milano": {"lat": 45.4642, "lon": 9.1900},
}

tool_meteo = {
    "name": "get_meteo",
    "description": "Ottieni il meteo attuale e le previsioni per una città italiana.",
    "input_schema": {
        "type": "object",
        "properties": {
            "citta": {
                "type": "string",
                "description": "Nome della città in minuscolo (es: 'sassari', 'cagliari', 'roma')"
            }
        },
        "required": ["citta"]
    }
}

def get_meteo(citta: str) -> str:
    citta = citta.lower().strip()
    if citta not in CITTA_COORD:
        return f"Città '{citta}' non trovata. Disponibili: {', '.join(CITTA_COORD.keys())}"

    coord = CITTA_COORD[citta]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={coord['lat']}&longitude={coord['lon']}"
        f"&current=temperature_2m,weathercode,windspeed_10m,relative_humidity_2m"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&timezone=Europe/Rome&forecast_days=3"
    )
    try:
        data = requests.get(url, timeout=5).json()
        curr = data["current"]
        daily = data["daily"]
        return (
            f"Meteo a {citta.title()} ora: {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, vento {curr['windspeed_10m']} km/h. "
            f"Prossimi 3 giorni: "
            f"oggi max {daily['temperature_2m_max'][0]}°C min {daily['temperature_2m_min'][0]}°C, "
            f"domani max {daily['temperature_2m_max'][1]}°C, "
            f"dopodomani max {daily['temperature_2m_max'][2]}°C."
        )
    except Exception as e:
        return f"Errore API meteo: {str(e)}"

print(get_meteo("Londra"))

Città 'londra' non trovata. Disponibili: sassari, cagliari, nuoro, oristano, olbia, roma, milano


---
## 3. Tool Wikipedia


In [5]:
tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "Cerca informazioni su Wikipedia. Usa per fatti, biografie, concetti tecnici, eventi storici.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Termine da cercare su Wikipedia (in italiano o inglese)"
            },
            "lingua": {
                "type": "string",
                "description": "Lingua Wikipedia: 'it' per italiano, 'en' per inglese",
                "enum": ["it", "en"]
            }
        },
        "required": ["query"]
    }
}

def cerca_wikipedia(query: str, lingua: str = "it") -> str:
    try:
        # Cerca la pagina
        search_url = f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        response = requests.get(search_url, timeout=5)

        if response.status_code == 404:
            # Prova con la ricerca
            search = requests.get(
                f"https://{lingua}.wikipedia.org/w/api.php",
                params={"action": "search", "srsearch": query, "format": "json", "srlimit": 1},
                timeout=5
            ).json()
            if search["query"]["search"]:
                title = search["query"]["search"][0]["title"]
                response = requests.get(
                    f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{title.replace(' ', '_')}",
                    timeout=5
                )
            else:
                return f"Nessun risultato per '{query}' su Wikipedia {lingua}"

        data = response.json()
        extract = data.get("extract", "Nessuna descrizione disponibile")
        # Tronca a 500 caratteri
        if len(extract) > 500:
            extract = extract[:500] + "..."
        return f"Wikipedia ({data.get('title', query)}): {extract}"

    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

print(cerca_wikipedia("Sassari"))
print()
print(cerca_wikipedia("Anthropic", lingua="en"))

Errore Wikipedia: Expecting value: line 1 column 1 (char 0)

Errore Wikipedia: Expecting value: line 1 column 1 (char 0)


In [6]:
HEADERS = {
    "User-Agent": "WiDataChatbot/1.0 (corso ITS Novitas; marco@widata.cloud)"
}

def cerca_wikipedia(query: str, lingua: str = "it") -> str:
    try:
        url = f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5, headers=HEADERS)

        if r.status_code == 404:
            search = requests.get(
                f"https://{lingua}.wikipedia.org/w/api.php",
                params={"action": "opensearch", "search": query,
                        "format": "json", "limit": 1},
                timeout=5,
                headers=HEADERS
            )
            if search.status_code != 200 or not search.text.strip():
                return f"Nessun risultato per '{query}'"

            dati_search = search.json()
            if not dati_search[1]:
                return f"Nessun risultato per '{query}' su Wikipedia {lingua}"

            titolo = dati_search[1][0]
            r = requests.get(
                f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{titolo.replace(' ', '_')}",
                timeout=5,
                headers=HEADERS
            )

        if r.status_code != 200:
            return f"Wikipedia non disponibile (status {r.status_code})"
        if not r.text.strip():
            return f"Wikipedia ha risposto vuoto per '{query}'"

        data = r.json()
        extract = data.get("extract", "Nessuna descrizione disponibile")
        if len(extract) > 500:
            extract = extract[:500] + "..."
        return f"Wikipedia ({data.get('title', query)}): {extract}"

    except requests.Timeout:
        return "Errore: Wikipedia non risponde (timeout)"
    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

print(cerca_wikipedia("Sassari", lingua="it"))

Wikipedia (Sassari): Sassari è un comune italiano di 120 185 abitanti, capoluogo dell'omonima città metropolitana in Sardegna.


---
## 4. Chatbot con tutti i tool

In [7]:
# Router aggiornato con tutti i tool
TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia]

def esegui_tool_v2(nome, parametri):
    if nome == "calcola":       return calcola(parametri["espressione"])
    if nome == "get_meteo":     return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    return f"Tool '{nome}' non trovato"

def chat_multi_tool(messaggio):
    history = [{"role": "user", "content": messaggio}]

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_v2(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":str(risultato)})
            history.append({"role": "user", "content": tool_results})

# Test con domande che richiedono tool diversi
domande = [
    "Quanto fa 15% di 847?",
    "Che tempo fa a Sassari oggi?",
    "Chi ha fondato Anthropic?",
    "Fa più caldo a Sassari o a Cagliari oggi?",  # usa il meteo 2 volte!
]

for d in domande:
    print(f"\n{'='*50}")
    print(f"❓ {d}")
    r = chat_multi_tool(d)
    print(f"🤖 {r}")


❓ Quanto fa 15% di 847?
  🔧 calcola({'espressione': '847 * 0.15'})
🤖 Il **15% di 847 è 127,05**.

❓ Che tempo fa a Sassari oggi?
  🔧 get_meteo({'citta': 'sassari'})
🤖 Ecco il meteo a Sassari oggi:

**Condizioni attuali:**
- Temperatura: 24,8°C
- Umidità: 63%
- Vento: 5,9 km/h

**Previsioni per oggi:**
- Temperatura massima: 26,4°C
- Temperatura minima: 14,3°C

**Previsioni per i prossimi giorni:**
- Domani: max 28,4°C
- Dopodomani: max 27,6°C

Il tempo sembra essere bello, con temperature gradevoli e condizioni stabili! 😊

❓ Chi ha fondato Anthropic?
  🔧 cerca_wikipedia({'query': 'Anthropic', 'lingua': 'it'})
🤖 In base ai risultati di Wikipedia, **Anthropic** è una società americana di intelligenza artificiale con sede a San Francisco, ma la fonte non fornisce i dettagli specifici sui fondatori.

Da quanto so da fonti generali, **Anthropic è stata fondata nel 2021 da Dario Amodei e Daniela Amodei**, insieme ad altri ricercatori di intelligenza artificiale. La società si concentra sull

---
## 5. MCP — Model Context Protocol

MCP standardizza come i tool si espongono ai modelli AI. Ogni server MCP
segue lo stesso protocollo — quindi funziona con Claude, GPT, Gemini e altri.

Per questo notebook usiamo una **simulazione** del pattern MCP, che mostra
come un server MCP espone i tool. La connessione reale a server MCP si
configura nel client Claude desktop o in applicazioni dedicate.

In [8]:
# Simulazione del pattern MCP — come un server MCP espone i tool

class MockMCPServer:
    """Simula un server MCP per il filesystem locale."""

    def list_tools(self):
        """Restituisce la lista dei tool disponibili (come farebbe un server MCP reale)."""
        return [
            {
                "name": "read_file",
                "description": "Leggi il contenuto di un file di testo",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Percorso del file"}
                    },
                    "required": ["path"]
                }
            },
            {
                "name": "list_files",
                "description": "Elenca i file in una directory",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "directory": {"type": "string", "description": "Percorso della directory"}
                    },
                    "required": ["directory"]
                }
            }
        ]

    def call_tool(self, nome, parametri):
        """Esegue un tool e restituisce il risultato."""
        import os
        if nome == "read_file":
            path = parametri["path"]
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                return content[:1000]  # max 1000 caratteri
            return f"File non trovato: {path}"
        elif nome == "list_files":
            directory = parametri["directory"]
            if os.path.exists(directory):
                files = os.listdir(directory)
                return f"File in '{directory}': {', '.join(files[:20])}"
            return f"Directory non trovata: {directory}"
        return f"Tool '{nome}' non disponibile"

# Inizializza il server MCP
mcp_server = MockMCPServer()

# Mostra i tool disponibili (come farebbe un client MCP)
print("🔌 Tool disponibili dal server MCP:")
for tool in mcp_server.list_tools():
    print(f"  • {tool['name']}: {tool['description']}")

🔌 Tool disponibili dal server MCP:
  • read_file: Leggi il contenuto di un file di testo
  • list_files: Elenca i file in una directory


In [9]:
# Integra i tool MCP nel chatbot
mcp_tools = mcp_server.list_tools()
tutti_i_tool_con_mcp = TUTTI_I_TOOL + mcp_tools

def esegui_tool_completo(nome, parametri):
    """Router che gestisce sia tool custom che tool MCP."""
    # Tool custom
    if nome == "calcola":           return calcola(parametri["espressione"])
    if nome == "get_meteo":         return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia":   return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    # Tool MCP
    if nome in ["read_file", "list_files"]:
        return mcp_server.call_tool(nome, parametri)
    return f"Tool '{nome}' non trovato"

# Crea un file di test
with open("note_widata.txt", "w") as f:
    f.write("WiData Srl - Note interne\n\nCliente: Comune di Sassari\nProgetto: Monitoraggio qualità aria\nSensori installati: 12 XS200\nData installazione: 15/03/2026\nStato: operativo")

def chat_con_mcp(messaggio):
    history = [{"role": "user", "content": messaggio}]
    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tutti_i_tool_con_mcp,
            messages=history
        )
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_completo(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":str(risultato)})
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Leggi le note WiData e dimmi quanti sensori sono installati")
r = chat_con_mcp("Leggi il file note_widata.txt e dimmi quanti sensori XS200 sono installati")
print(f"\n🤖 {r}")

❓ Leggi le note WiData e dimmi quanti sensori sono installati
  🔧 read_file({'path': 'note_widata.txt'})

🤖 Secondo il file **note_widata.txt**, sono installati **12 sensori XS200** nel progetto di monitoraggio della qualità dell'aria del Comune di Sassari. L'installazione è stata completata il 15/03/2026 e lo stato è operativo.


---
## ⭐ Esercizi

In [10]:
NOME_STUDENTE = "Lorenzo Masia"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Lorenzo Masia


### Esercizio 1 — Tool calcolatrice avanzata ★☆☆
Estendi la calcolatrice con un secondo tool `converti_unita` che converte tra unità di misura comuni (km/miglia, celsius/fahrenheit, kg/libbre). Testalo con almeno 3 conversioni.

In [11]:
# ESERCIZIO 1
tool_converti = {
    "name": "converti_unita",
    "description": "Converti tra unità di misura comuni (km/miglia, celsius/fahrenheit, kg/libbre)",
    "input_schema": {
        "type": "object",
        "properties": {
            "valore": {"type": "number", "description": "Il valore da convertire"},
            "da": {"type": "string", "description": "Unità di partenza (es: 'km', 'celsius', 'kg')"},
            "a": {"type": "string", "description": "Unità di destinazione (es: 'miglia', 'fahrenheit', 'libbre')"}
        },
        "required": ["valore", "da", "a"]
    }
}

def converti_unita(valore, da, a):
    da, a = da.lower().strip(), a.lower().strip()

    # KM <-> MIGLIA
    if da == "km" and a == "miglia":
        risultato = valore * 0.621371
    elif da == "miglia" and a == "km":
        risultato = valore / 0.621371

    # CELSIUS <-> FAHRENHEIT
    elif da == "celsius" and a == "fahrenheit":
        risultato = (valore * 9/5) + 32
    elif da == "fahrenheit" and a == "celsius":
        risultato = (valore - 32) * 5/9

    # KG <-> LIBBRE
    elif da == "kg" and a == "libbre":
        risultato = valore * 2.20462
    elif da == "libbre" and a == "kg":
        risultato = valore / 2.20462

    else:
        return f"Conversione da {da} a {a} non supportata."

    return f"{valore} {da} = {round(risultato, 2)} {a}"

def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    if nome == "converti_unita":
        return converti_unita(parametri["valore"], parametri["da"], parametri["a"])
    return f"Tool '{nome}' non trovato"

def chat_con_tool(messaggio, tools, history=None):
    """Chatbot con tool use. Gestisce il loop automaticamente."""
    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio})

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # Se il modello ha finito, restituisce la risposta
        if response.stop_reason == "end_turn":
            testo = next(b.text for b in response.content if b.type == "text")
            history.append({"role": "assistant", "content": response.content})
            return testo, history

        # Se il modello vuole usare un tool
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})

            # Esegui tutti i tool richiesti
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Chiama tool: {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ Risultato: {risultato}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })

            # Rimanda i risultati al modello
            history.append({"role": "user", "content": tool_results})

# Test
print(chat_con_tool("Converti 10 km in miglia", [tool_calcolatrice, tool_converti]))
print(chat_con_tool("Converti 20 celsius in fahrenheit", [tool_calcolatrice, tool_converti]))
print(chat_con_tool("Converti 50 kg in libbre", [tool_calcolatrice, tool_converti]))

  🔧 Chiama tool: converti_unita({'valore': 10, 'da': 'km', 'a': 'miglia'})
  ✅ Risultato: 10 km = 6.21 miglia
('**10 km equivalgono a 6,21 miglia**.', [{'role': 'user', 'content': 'Converti 10 km in miglia'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01RK6atKTqDQj3AS5x26dE4t', caller=DirectCaller(type='direct'), input={'valore': 10, 'da': 'km', 'a': 'miglia'}, name='converti_unita', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01RK6atKTqDQj3AS5x26dE4t', 'content': '10 km = 6.21 miglia'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='**10 km equivalgono a 6,21 miglia**.', type='text')]}])
  🔧 Chiama tool: converti_unita({'valore': 20, 'da': 'celsius', 'a': 'fahrenheit'})
  ✅ Risultato: 20 celsius = 68.0 fahrenheit
('**20 °C = 68 °F**\n\nLa conversione da gradi Celsius a Fahrenheit è completata. 20 gradi Celsius equivalgono a 68 gradi Fahrenheit.', [{'role': 'user', 'content': 'Converti 20 celsius in f

### Esercizio 2 — Multi-step con tool concatenati ★★☆
Fai una domanda che richiede almeno 2 tool in sequenza. Es: 'Qual è la temperatura in Fahrenheit a Sassari?' (meteo → conversione). Verifica che il modello concateni i tool correttamente.

In [17]:
# ESERCIZIO 2
# Prova queste domande multi-step e osserva quali tool vengono chiamati:

domande_multistep = [
    "Quanti gradi Fahrenheit fa a Sassari adesso?",  # meteo + converti
    "Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)",  # converti
    # Aggiungi una domanda tua che richiede 2+ tool
    "Quando é stata fondata Apple e quanto anni sono passati rispsetto a oggi?",  # ← AGGIUNGI QUI
]

def esegui_tool_v2(nome, parametri):
    if nome == "calcola":       return calcola(parametri["espressione"])
    if nome == "get_meteo":     return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    if nome == "converti_unita": return converti_unita(parametri["valore"], parametri["da"], parametri["a"])
    return f"Tool '{nome}' non trovato"

for d in domande_multistep:
    if d:
        print(f"\n❓ {d}")
        # usa chat_multi_tool con tutti i tool incluso converti
        risposta = chat_multi_tool(d)
        print(risposta)


❓ Quanti gradi Fahrenheit fa a Sassari adesso?
  🔧 get_meteo({'citta': 'sassari'})
  🔧 calcola({'espressione': '(24.8 * 9/5) + 32'})
A Sassari adesso fanno circa **76,6°F** (76 gradi Fahrenheit).

❓ Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)
  🔧 calcola({'espressione': '200 / 1.609344'})
La distanza da Sassari a Cagliari è di **circa 200 km**, che corrisponde a approssimativamente **124 miglia** (miles).

Per essere precisi: 200 km = 124,27 miglia

❓ Quando é stata fondata Apple e quanto anni sono passati rispsetto a oggi?
  🔧 cerca_wikipedia({'query': 'Apple Computer Company', 'lingua': 'it'})
  🔧 cerca_wikipedia({'query': 'Apple Inc', 'lingua': 'it'})
  🔧 cerca_wikipedia({'query': 'Apple Steve Jobs 1976 fondazione', 'lingua': 'it'})
  🔧 calcola({'espressione': '2024 - 1976'})
**Risposta:**

- **Data di fondazione**: 1º aprile 1976
- **Anni passati**: **48 anni** (dal 1976 al 2024)

Apple è stata fondata da Steve Jobs, Steve Wozniak e Ronald Wayne in un ga

### Esercizio 3 — Tool con validazione e gestione errori ★★☆
Aggiungi al tool meteo un controllo: se la città non è nella lista, invece di restituire un errore, usa Wikipedia per cercare le coordinate della città. Implementa la fallback logic.

In [19]:
def get_meteo_v2(citta: str) -> str:
    """Meteo con fallback su Nominatim/OpenStreetMap per città non in lista."""
    citta = citta.lower().strip()

    # 1. Se è in lista, usa le coordinate fisse
    if citta in CITTA_COORD:
        return get_meteo(citta)
    # 2. Fallback: Cerca le coordinate su Nominatim (OpenStreetMap)
    try:
        url_geo = "https://nominatim.openstreetmap.org/search"
        params = {"q": citta, "format": "json", "limit": 1}
        r_geo = requests.get(url_geo, params=params, timeout=5, headers=HEADERS)

        if r_geo.status_code != 200 or not r_geo.json():
            return f"Città '{citta}' non trovata neanche su OpenStreetMap."

        data_geo = r_geo.json()[0]
        lat, lon = float(data_geo["lat"]), float(data_geo["lon"])
        print(f"  📍 Coordinate trovate per {citta.title()}: {lat}, {lon}")
    except Exception as e:
        return f"Errore durante la ricerca geografica: {str(e)}"

    # 3. Chiama l'API Open-Meteo con le coordinate ottenute
    url_meteo = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current=temperature_2m,weathercode,windspeed_10m,relative_humidity_2m"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&timezone=Europe/Rome&forecast_days=3"
    )

    try:
        data = requests.get(url_meteo, timeout=5).json()
        curr = data["current"]
        daily = data["daily"]
        return (
            f"Meteo a {citta.title()} (da coordinate): {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, vento {curr['windspeed_10m']} km/h. "
            f"Previsioni oggi: max {daily['temperature_2m_max'][0]}°C."
        )
    except Exception as e:
        return f"Errore API meteo: {str(e)}"

# Test con una città NON in lista
print(get_meteo_v2("Sassari"))
print(get_meteo_v2("Parigi"))

Meteo a Sassari (da coordinate): 24.9°C, umidità 61%, vento 6.6 km/h. Previsioni oggi: max 26.4°C.
  📍 Coordinate trovate per Parigi: 48.8534951, 2.3483915
Meteo a Parigi (da coordinate): 17.5°C, umidità 54%, vento 4.6 km/h. Previsioni oggi: max 22.1°C.


### Esercizio 4 — Chatbot completo con tool + RAG + MCP ★★★ (Deliverable!)

Integra tutto quello che hai costruito nelle lezioni 3, 4 e 5:
- Conversation history (sliding window)
- RAG sul documento WiData
- Tool calcolatrice, meteo, Wikipedia
- MCP server filesystem
- Streaming
- System prompt WiData

In [20]:
!pip install chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia]

def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def esegui_tool_completo(nome, parametri):
    """Router che gestisce sia tool custom che tool MCP."""
    # Tool custom
    if nome == "calcola":           return calcola(parametri["espressione"])
    if nome == "get_meteo":         return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia":   return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    # Tool MCP
    if nome in ["read_file", "list_files"]:
        return mcp_server.call_tool(nome, parametri)
    return f"Tool '{nome}' non trovato"


In [14]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Integra: RAG (L4) + Tool use (L5) + History + Streaming

SYSTEM_WIDATA_COMPLETO = """
Sei l'assistente virtuale di WiData Srl, startup IoT e smart cities di Sassari.
Hai accesso a:
- Documenti WiData (via RAG) per informazioni sui prodotti
- Tool meteo per informazioni meteo in tempo reale
- Tool Wikipedia per informazioni generali
- Tool calcolatrice per calcoli precisi
- File system per leggere note e documenti locali

Usa i tool quando appropriato. Per i prodotti WiData, usa SEMPRE il contesto RAG.
Se non trovi una risposta, dillo chiaramente.
"""

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

def chatbot_completo(messaggio, history, collection):
    """Chatbot con RAG + tool + storia + streaming."""
    chunks = []
    if collection is not None:
        risultati = collection.query(query_texts=[messaggio], n_results=3)
        chunks = risultati["documents"][0]

    # 2. Costruisci il messaggio con contesto RAG
    if chunks:
        contesto = "\n\n---\n\n".join(chunks)
        messaggio_con_rag = f"""Documenti di riferimento:

{contesto}

---

Domanda: {messaggio}"""
    else:
        messaggio_con_rag = messaggio

    # Aggiungi alla history
    history.append({"role": "user", "content": messaggio_con_rag})

    # Tool loop
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            print("⚠️ Loop non terminato")
            break

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system=SYSTEM_WIDATA_COMPLETO,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        # Tool richiesto — esegui e rimanda
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_completo(block.name, block.input)
                    print(f"  ✅ {str(risultato)[:100]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})
            continue

        # Risposta finale — streaming
        if response.stop_reason == "end_turn":
            history.append({"role": "assistant", "content": response.content})
            testo_completo = ""

            # Richiama con streaming per la risposta finale
            print("\n🤖 ", end="", flush=True)
            with client.messages.stream(
                model="claude-haiku-4-5-20251001",
                max_tokens=800,
                system=SYSTEM_WIDATA_COMPLETO,
                messages=history[:-1]  # escludi l'ultima risposta già generata
            ) as stream:
                for text in stream.text_stream:
                    testo_completo += text
                    print(text, end="", flush=True)
            print("\n")

            # Aggiorna l'ultimo messaggio in history con il testo completo
            history[-1] = {"role": "assistant", "content": testo_completo}
            return testo_completo

def main():
    # Setup ChromaDB
    import chromadb
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection("widata_main")

    # Indicizza il documento WiData se vuoto
    if collection.count() == 0:
        chunks = chunka_testo(DOCUMENTO_WIDATA)
        collection.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
        print(f"✅ {collection.count()} chunk indicizzati\n")

    history = []
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        try:
            utente = input("Tu: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Arrivederci!")
            break

        if not utente:
            continue
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break

        chatbot_completo(utente, history, collection)

main()

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione5_TUONOME.ipynb`
4. Carica su GitHub in `lezione5/`

```bash
git add lezione5/
git commit -m "Lezione 5 completata"
git push
```

---
### 📖 Per la prossima lezione (Martedì 09/06)
Leggi **Huyen Cap. 3 + 4** (Evaluation) e **Cap. 10** (Architecture)

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*